# Test Postgre

## Init

In [ ]:
# ========== Init: auto-detect PostgreSQL path (version-matched) ==========
import subprocess, time, shutil, os, glob
from pathlib import Path

# Auto-detect PostgreSQL path
PSQL = shutil.which("psql")
if not PSQL:
    raise FileNotFoundError("psql not found, please install PostgreSQL client")

def _get_pg_version(data_dir: str) -> str:
    """Read the PG major version from the data directory."""
    vf = os.path.join(data_dir, "PG_VERSION")
    if os.path.isfile(vf):
        with open(vf) as f:
            return f.read().strip()
    return None

def _find_pg_binary(name: str, data_dir: str = None) -> str:
    """Find a pg tool matching the data directory version."""
    # 1. Infer search paths from data directory version
    search_dirs = []
    if data_dir:
        ver = _get_pg_version(data_dir)
        if ver:
            # Search paths containing version: pgsql-13.1, postgresql-13, etc.
            patterns = [
                f"/home/*/pgsql-{ver}.*/bin",
                f"/usr/local/pgsql/{ver}*/bin",
                f"/usr/local/pgsql-{ver}*/bin",
                f"/usr/lib/postgresql/{ver}/bin",
            ]
            for pat in patterns:
                for d in sorted(glob.glob(pat)):
                    if os.path.isdir(d):
                        search_dirs.append(d)
    # 2. Found via shutil.which (verify version compatibility)
    found = shutil.which(name)
    if found:
        search_dirs.append(str(Path(found).parent))
    # 3. Extra paths
    for d in ["/usr/local/pgsql/13.1/bin", "/usr/local/pgsql/16/bin", "/usr/lib/postgresql/16/bin"]:
        if os.path.isdir(d):
            search_dirs.append(d)
    for d in search_dirs:
        candidate = os.path.join(d, name)
        if os.path.isfile(candidate) and os.access(candidate, os.X_OK):
            return candidate
    raise FileNotFoundError(f"Cannot find a version-matched {name}")

# Get data_directory from running PG
r = subprocess.run(
    [PSQL, "-U", "postgres", "-t", "-A", "-c", "SHOW data_directory;"],
    capture_output=True, text=True
)
if r.returncode == 0:
    PG_DATA = r.stdout.strip()
else:
    PG_DATA = os.environ.get("PGDATA", "/home/liwei/pgdata")

PG_CTL = _find_pg_binary("pg_ctl", PG_DATA)

print(f"PSQL:     {PSQL}")
print(f"PG_CTL:   {PG_CTL}")
print(f"PG_DATA:  {PG_DATA}")
if PG_DATA:
    print(f"PG Version:  {_get_pg_version(PG_DATA)}")


def cold_restart_pg():
    """Cold restart PG: stop PG -> clear OS page cache -> start PG (using version-matched pg_ctl)."""
    t0 = time.perf_counter()
    # 1. Stop PG
    subprocess.run([PG_CTL, "-D", PG_DATA, "-m", "fast", "stop"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    # 2. Clear OS page cache
    try:
        with open("/proc/sys/vm/drop_caches", "w") as f:
            f.write("3\n")
    except PermissionError:
        pass
    # 3. Start PG (version-matched, will not fail)
    subprocess.run([PG_CTL, "-D", PG_DATA, "start"],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    # 4. Wait for ready
    for i in range(120):
        r = subprocess.run(
            [PSQL, "-U", "postgres", "-t", "-A", "-c", "SELECT 1;"],
            capture_output=True, text=True
        )
        if r.returncode == 0:
            elapsed = time.perf_counter() - t0
            print(f"  Cold restart complete ({elapsed:.1f}s, waited {i}s)")
            return elapsed
        time.sleep(1)
    raise RuntimeError("PostgreSQL startup timeout (not ready after 120 seconds)")


print("Initialization complete\n")


In [ ]:
import os
import sys
import re
import time
import subprocess
import shlex
import csv
from pathlib import Path

# Auto-detect psql path (prefer the PSQL variable already detected in the first cell)
try:
    PSQL_BIN = PSQL
except NameError:
    import shutil
    PSQL_BIN = shutil.which("psql")
    if not PSQL_BIN:
        PSQL_BIN = "/usr/local/pgsql/13.1/bin/psql"  # fallback

# TO CHANGE: statistics collection parallelism
PG_PARALLELISM = 8

# Add scripts directory to path
sys.path.append(os.path.abspath('../scripts'))

from extract_card_from_pg_plan import extract_cardinalities

project_root = Path('..').resolve()
benchmark_dir = project_root / "Benchmark" / "workloads"

running_space = Path("./running_space").resolve()
running_space.mkdir(parents=True, exist_ok=True)

checkpoint_dir = Path("./checkpoint/Postgre").resolve()
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# PostgreSQL connection info (can be overridden via env vars)
PG_CONN_BASE = os.environ.get(
    "PG_CONN_BASE",
    f"host=127.0.0.1 port=5432 user={os.environ.get('USER', 'postgres')}"
)

def make_conn_str(db_name: str) -> str:
    return f"{PG_CONN_BASE} dbname={db_name}"


BENCHMARKS = {
    "STATS": {
        "db_name": "stats",
        "queries_file": benchmark_dir / "STATS-CEB" / "queries.sql",
        "subquery_file": benchmark_dir / "STATS-CEB" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_stats.txt",
        "explain_output": running_space / "pg_stats_explain.txt",
        "explain_queries_output": running_space / "pg_stats_queries_explain.txt",
    },
    "JOBLight": {
        "db_name": "imdblight",
        "queries_file": benchmark_dir / "JOBLight" / "queries.sql",
        "subquery_file": benchmark_dir / "JOBLight" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_joblight.txt",
        "explain_output": running_space / "pg_joblight_explain.txt",
        "explain_queries_output": running_space / "pg_joblight_queries_explain.txt",
    },
    "JOBLightRanges": {
        "db_name": "imdblightranges",
        "queries_file": benchmark_dir / "JOBLightRanges" / "queries.sql",
        "subquery_file": benchmark_dir / "JOBLightRanges" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_joblr.txt",
        "explain_output": running_space / "pg_joblr_explain.txt",
        "explain_queries_output": running_space / "pg_joblr_queries_explain.txt",
    },
    "JOBM": {
        "db_name": "imdbm",
        "queries_file": benchmark_dir / "JOBM" / "queries.sql",
        "subquery_file": benchmark_dir / "JOBM" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_jobm.txt",
        "explain_output": running_space / "pg_jobm_explain.txt",
        "explain_queries_output": running_space / "pg_jobm_queries_explain.txt",
    },
    "StatsJoin": {
        "db_name": "stats",
        "queries_file": benchmark_dir / "StatsJoin" / "queries.sql",
        "subquery_file": benchmark_dir / "StatsJoin" / "subquery" / "subquery.sql",
        "card_output": checkpoint_dir / "card_statsjoin.txt",
        "explain_output": running_space / "pg_statsjoin_explain.txt",
        "explain_queries_output": running_space / "pg_statsjoin_queries_explain.txt",
    },
}

print("Configuration complete, ready to collect statistics and cardinality estimation results")

In [ ]:
def prepare_explain_sql(queries_file: Path, output_file: Path) -> None:
    """
    Copy queries file to output_file, adding EXPLAIN before each SELECT statement.
    """
    if not queries_file.exists():
        raise FileNotFoundError(f"Cannot find queries file: {queries_file}")

    content = queries_file.read_text(encoding="utf-8")
    content = re.sub(r"count\s*\(\s*\*\s*\)", "*", content, flags=re.IGNORECASE)
    pattern = r"^(\s*)(select\s+)"
    replacement = r"\1EXPLAIN \2"
    new_content = re.sub(pattern, replacement, content, flags=re.IGNORECASE | re.MULTILINE)

    output_file.write_text(new_content, encoding="utf-8")


def run_psql_sql(conn_str: str, sql: str) -> str:
    cmd = [
        PSQL_BIN,
        "-v",
        "ON_ERROR_STOP=1",
        conn_str,
        "-t",
        "-A",
        "-c",
        sql
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "psql execution failed")
    return result.stdout


def run_psql_file(conn_str: str, sql_file: Path, output_file: Path) -> None:
    cmd = [
        PSQL_BIN,
        "-v",
        "ON_ERROR_STOP=1",
        conn_str,
        "-f",
        str(sql_file),
        "-o",
        str(output_file)
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or "psql execution failed")


def _get_user_tables(conn_str: str):
    """Get all user tables (skip _ss1d0 sample tables), returns [(schema, table), ...]."""
    sql = (
        "SELECT schemaname, tablename FROM pg_tables "
        "WHERE schemaname NOT IN ('pg_catalog', 'information_schema') "
        "  AND strpos(tablename, '_ss1d0') = 0"
    )
    output = run_psql_sql(conn_str, sql)
    tables = []
    for line in output.strip().splitlines():
        if not line.strip():
            continue
        parts = line.strip().split('|')
        if len(parts) >= 2:
            tables.append((parts[0].strip(), parts[1].strip()))
    return tables


def _find_unindexed_column(conn_str: str, schema: str, table: str) -> str:
    """Find an unindexed column on the table, used for ORDER BY to trigger full table scan warmup."""
    # Get all columns
    cols_sql = (
        f"SELECT a.attname FROM pg_attribute a "
        f"JOIN pg_class c ON a.attrelid = c.oid "
        f"JOIN pg_namespace n ON c.relnamespace = n.oid "
        f"WHERE n.nspname = '{schema}' AND c.relname = '{table}' "
        f"  AND a.attnum > 0 AND NOT a.attisdropped "
        f"ORDER BY a.attnum"
    )
    all_cols = run_psql_sql(conn_str, cols_sql).strip().splitlines()
    if not all_cols:
        return "id"

    # Get indexed columns
    idx_sql = (
        f"SELECT indexdef FROM pg_indexes "
        f"WHERE schemaname = '{schema}' AND tablename = '{table}'"
    )
    idx_defs = run_psql_sql(conn_str, idx_sql).strip().splitlines()
    indexed_cols = set()
    for idx_def in idx_defs:
        m = re.search(r"\(([^)]+)\)", idx_def)
        if m:
            for col in m.group(1).split(','):
                indexed_cols.add(col.strip())

    for col in all_cols:
        if col not in indexed_cols:
            return col
    # fallback: last column
    return all_cols[-1]


def collect_pg_statistics(conn_str: str, run_vacuum: bool = False):
    """
    Collect PostgreSQL statistics and return time (seconds), warm-start baseline.
    Process: cold restart PG -> per-table SELECT * ORDER BY warmup -> per-table ANALYZE timed.
    """
    # 1. Cold start
    cold_restart_pg()

    # 2. Get user tables
    tables = _get_user_tables(conn_str)
    print(f"  {len(tables)} tables to ANALYZE")

    # 3. Per-table: warmup (not timed) -> ANALYZE (timed)
    total_elapsed = 0.0
    for schema, table in tables:
        # Find unindexed column
        col = _find_unindexed_column(conn_str, schema, table)
        # Warmup: SELECT * ORDER BY col LIMIT 5 triggers full scan, loads data into OS cache
        run_psql_sql(
            conn_str,
            f'SELECT * FROM "{schema}"."{table}" ORDER BY "{col}" LIMIT 5'
        )
        # Timed ANALYZE
        start = time.perf_counter()
        run_psql_sql(conn_str, f'ANALYZE "{schema}"."{table}"')
        elapsed = time.perf_counter() - start
        total_elapsed += elapsed
        print(f"    {schema}.{table}: {elapsed:.3f}s (prewarmed via ORDER BY {col})")

    print(f"  Per-table ANALYZE total: {total_elapsed:.3f}s")

    if run_vacuum:
        run_psql_sql(conn_str, "VACUUM FULL pg_statistic;")
        run_psql_sql(conn_str, "VACUUM FULL pg_statistic_ext_data;")
    return total_elapsed


def get_pg_statistics_size(conn_str: str):
    """Get PostgreSQL statistics size (bytes)."""
    size_query = (
        "SELECT pg_total_relation_size('pg_statistic') "
        "+ pg_total_relation_size('pg_statistic_ext_data');"
    )
    try:
        output = run_psql_sql(conn_str, size_query)
        return int(output.strip().splitlines()[-1])
    except Exception:
        try:
            output = run_psql_sql(conn_str, "SELECT pg_total_relation_size('pg_statistic');")
            return int(output.strip().splitlines()[-1])
        except Exception as e:
            raise RuntimeError(f"Failed to get statistics size: {e}") from e


def extract_pg_cardinalities(conn_str: str, subquery_file: Path, explain_output: Path):
    """Generate EXPLAIN SQL for subquery file, run psql, extract cardinality estimates."""
    explain_sql = running_space / "explain.sql"
    prepare_explain_sql(subquery_file, explain_sql)
    run_psql_file(conn_str, explain_sql, explain_output)
    return extract_cardinalities(explain_output)


def time_explain_queries(conn_str: str, queries_file: Path, explain_output: Path) -> float:
    """Generate EXPLAIN SQL for main query file, run psql, return wall-clock time (seconds)."""
    explain_sql = running_space / "explain_queries.sql"
    prepare_explain_sql(queries_file, explain_sql)
    start = time.perf_counter()
    run_psql_file(conn_str, explain_sql, explain_output)
    return time.perf_counter() - start


In [ ]:
def evaluate_pg_benchmark(benchmark_name: str, collect_stats: bool = True, run_vacuum: bool = False):
    if benchmark_name not in BENCHMARKS:
        raise ValueError(f"Unsupported benchmark: {benchmark_name}")

    cfg = BENCHMARKS[benchmark_name]
    conn_str = make_conn_str(cfg["db_name"])

    print(f"\n{'=' * 60}")
    print(f"Benchmark: {benchmark_name}")
    print(f"Database: {cfg['db_name']}")
    if cfg["queries_file"] is not None:
        print(f"Main query file: {cfg['queries_file']}")
    if cfg["subquery_file"] is not None:
        print(f"Subquery file: {cfg['subquery_file']}")

    stats_time = None
    stats_size = None
    if collect_stats:
        stats_time = collect_pg_statistics(conn_str, run_vacuum=run_vacuum)
    stats_size = get_pg_statistics_size(conn_str)

    cardinals = None
    if cfg["subquery_file"] is not None:
        cardinals = extract_pg_cardinalities(
            conn_str,
            cfg["subquery_file"],
            cfg["explain_output"]
        )
        cfg["card_output"].write_text("\n".join(str(c) for c in cardinals), encoding="utf-8")

    eval_time = None
    if cfg["queries_file"] is not None:
        eval_time = time_explain_queries(
            conn_str,
            cfg["queries_file"],
            cfg["explain_queries_output"]
        )

    print("\nResults:")
    print(f"  Statistics collection time (warm start, per-table prewarmed): {stats_time}  seconds")
    print(f"  Statistics size: {stats_size} bytes")
    if eval_time is not None:
        print(f"  Main query EXPLAIN time: {eval_time}  seconds")
    if cardinals is not None:
        print(f"  Cardinality count: {len(cardinals)}")

    return {
        "stats_time": stats_time,
        "stats_size": stats_size,
        "eval_time": eval_time,
        "cardinalities": cardinals
    }

In [ ]:
results = {}
for name in BENCHMARKS:
    results[name] = evaluate_pg_benchmark(name, collect_stats=True, run_vacuum=False)

In [ ]:
stats_summary = {
    name: {
        "stats_time": results[name]["stats_time"],
        "stats_size": results[name]["stats_size"],
        "eval_time": results[name]["eval_time"],
    }
    for name in results
}

stats_summary_path = checkpoint_dir / "pg_stats_summary.csv"
with stats_summary_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["Benchmark", "BuildTime", "StatisticsSize", "EvaluationTime"])
    for name, info in stats_summary.items():
        writer.writerow([name, info["stats_time"], info["stats_size"], info["eval_time"] or ""])
print(f"Statistics summary saved: {stats_summary_path}")